# Dataset exploration and feature engineering

This notebook documents the preprocessing step used to prepare the NASA battery-cycle data for the dimming-control simulation pipeline.

It can be used in two ways:

- If the raw NASA/Kaggle dataset files are available under `data/raw/`, the notebook extracts the engineered per-cycle feature table.
- If the raw files are not present, the notebook loads the included processed table at `data/processed/engineered_metadata.csv` and verifies its structure.

The main output is:

`data/processed/engineered_metadata.csv`

# Setup and paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "engineered_metadata.csv"

print("Repository root:", REPO_ROOT)
print("Raw data directory:", RAW_DIR)
print("Processed output:", PROCESSED_PATH)

# Locate raw dataset files

The Kaggle conversion of the NASA dataset commonly contains a `metadata.csv` file and a `data/` folder with per-cycle CSV traces. This cell searches common layouts under `data/raw/`.

In [ ]:
candidate_roots = [
    RAW_DIR / "cleaned_dataset",
    RAW_DIR / "nasa-battery-dataset",
    RAW_DIR / "nasa_battery_dataset",
    RAW_DIR,
]

metadata_path = None
cycle_data_dir = None

for root in candidate_roots:
    maybe_meta = root / "metadata.csv"
    maybe_data = root / "data"

    if maybe_meta.exists():
        metadata_path = maybe_meta

    if maybe_data.exists():
        cycle_data_dir = maybe_data

    if metadata_path is not None and cycle_data_dir is not None:
        break

raw_available = metadata_path is not None and cycle_data_dir is not None

print("Metadata path:", metadata_path)
print("Cycle data directory:", cycle_data_dir)
print("Raw dataset available:", raw_available)

# Metadata overview

In [ ]:
if raw_available:
    metadata = pd.read_csv(metadata_path)
    print("Raw metadata shape:", metadata.shape)
    display(metadata.head())

    if "type" in metadata.columns:
        cycle_counts = metadata["type"].value_counts(dropna=False)
        display(cycle_counts)

        ax = cycle_counts.plot(kind="bar", figsize=(7, 4), title="Cycle type distribution")
        ax.set_xlabel("Cycle type")
        ax.set_ylabel("Count")
        plt.tight_layout()
        plt.show()

    if "battery_id" in metadata.columns:
        battery_counts = metadata["battery_id"].value_counts().sort_index()
        print("Number of batteries:", battery_counts.shape[0])
        display(battery_counts.head())
else:
    print("Raw metadata not found. The notebook will use the included processed feature table.")

# Feature extraction

The feature table is built at the discharge-cycle level. Each row corresponds to one discharge cycle and includes summary statistics from voltage, current, temperature, and time traces.

The feature names are aligned with the main simulation notebook.

In [ ]:
TARGET = "cycle_duration"
GROUP_COL = "battery_id"

FEATURES = [
    "avg_voltage", "voltage_drop", "voltage_gradient", "voltage_std",
    "avg_current", "max_current", "current_std",
    "initial_temp", "final_temp", "temp_rise",
    "energy_approx", "start_voltage", "end_voltage",
]

def first_existing_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def read_cycle_file(filename):
    path = cycle_data_dir / str(filename)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

def extract_discharge_features(row):
    df = read_cycle_file(row["filename"])

    voltage_col = first_existing_column(df, ["Voltage_measured", "voltage_measured", "voltage", "Voltage"])
    current_col = first_existing_column(df, ["Current_measured", "current_measured", "current", "Current"])
    temp_col = first_existing_column(df, ["Temperature_measured", "temperature_measured", "temperature", "Temperature"])
    time_col = first_existing_column(df, ["Time", "time"])

    required = {
        "voltage": voltage_col,
        "current": current_col,
        "temperature": temp_col,
        "time": time_col,
    }

    missing = [name for name, col in required.items() if col is None]
    if missing:
        raise ValueError(f"Missing required trace columns {missing} in {row['filename']}")

    voltage = pd.to_numeric(df[voltage_col], errors="coerce").to_numpy(dtype=float)
    current = pd.to_numeric(df[current_col], errors="coerce").to_numpy(dtype=float)
    temp = pd.to_numeric(df[temp_col], errors="coerce").to_numpy(dtype=float)
    time = pd.to_numeric(df[time_col], errors="coerce").to_numpy(dtype=float)

    valid = np.isfinite(voltage) & np.isfinite(current) & np.isfinite(temp) & np.isfinite(time)
    voltage, current, temp, time = voltage[valid], current[valid], temp[valid], time[valid]

    if len(time) < 2:
        raise ValueError(f"Not enough valid samples in {row['filename']}")

    cycle_duration = float(np.nanmax(time) - np.nanmin(time))
    if cycle_duration <= 0:
        cycle_duration = float(np.nanmax(time))

    dt = np.diff(time, prepend=time[0])
    dt = np.where(np.isfinite(dt) & (dt >= 0), dt, 0.0)

    power = np.abs(voltage * current)
    energy_approx = float(np.sum(power * dt))

    return {
        "uid": row.get("uid", row.name),
        "battery_id": str(row.get("battery_id")),
        "filename": row.get("filename"),
        "type": row.get("type", "discharge"),
        "ambient_temperature": row.get("ambient_temperature", np.nan),
        "capacity": row.get("capacity", np.nan),
        "cycle_duration": cycle_duration,
        "avg_voltage": float(np.nanmean(voltage)),
        "voltage_drop": float(voltage[0] - voltage[-1]),
        "voltage_gradient": float((voltage[-1] - voltage[0]) / max(cycle_duration, 1e-9)),
        "voltage_std": float(np.nanstd(voltage)),
        "avg_current": float(np.nanmean(current)),
        "max_current": float(np.nanmax(np.abs(current))),
        "current_std": float(np.nanstd(current)),
        "initial_temp": float(temp[0]),
        "final_temp": float(temp[-1]),
        "temp_rise": float(temp[-1] - temp[0]),
        "energy_approx": energy_approx,
        "start_voltage": float(voltage[0]),
        "end_voltage": float(voltage[-1]),
    }

# Build or load the engineered feature table

In [ ]:
if raw_available:
    discharge_meta = metadata.copy()

    if "type" in discharge_meta.columns:
        discharge_meta = discharge_meta[
            discharge_meta["type"].astype(str).str.lower().eq("discharge")
        ].copy()

    discharge_meta = discharge_meta.reset_index(drop=True)
    print("Discharge cycles found:", len(discharge_meta))

    records = []
    failures = []

    for idx, row in discharge_meta.iterrows():
        try:
            records.append(extract_discharge_features(row))
        except Exception as exc:
            failures.append((row.get("filename", idx), str(exc)))

    engineered_metadata = pd.DataFrame(records)

    print("Extracted rows:", engineered_metadata.shape)
    print("Failed files:", len(failures))

    if failures:
        display(pd.DataFrame(failures, columns=["filename", "error"]).head(10))

    engineered_metadata.to_csv(PROCESSED_PATH, index=False)
    print("Saved:", PROCESSED_PATH)

else:
    if not PROCESSED_PATH.exists():
        raise FileNotFoundError(
            f"Raw files were not found and processed table is missing: {PROCESSED_PATH}"
        )

    engineered_metadata = pd.read_csv(PROCESSED_PATH)
    print("Loaded included processed table:", engineered_metadata.shape)

# Processed table checks

In [ ]:
required_columns = ["uid", GROUP_COL, TARGET] + FEATURES
missing = [col for col in required_columns if col not in engineered_metadata.columns]

if missing:
    raise ValueError(f"Processed table is missing required columns: {missing}")

print("Processed table shape:", engineered_metadata.shape)
print("Batteries:", engineered_metadata[GROUP_COL].astype(str).nunique())
print("Target mean:", round(pd.to_numeric(engineered_metadata[TARGET], errors="coerce").mean(), 3))
print("Target std:", round(pd.to_numeric(engineered_metadata[TARGET], errors="coerce").std(), 3))

display(engineered_metadata.head())

# Quick exploratory plots

These plots are included only as lightweight sanity checks of the processed feature table.

In [ ]:
ax = engineered_metadata[TARGET].plot(
    kind="hist",
    bins=30,
    figsize=(7, 4),
    title="Distribution of discharge cycle duration"
)
ax.set_xlabel("Cycle duration")
plt.tight_layout()
plt.show()

counts = engineered_metadata[GROUP_COL].astype(str).value_counts().sort_index()
ax = counts.plot(kind="bar", figsize=(10, 4), title="Discharge cycles per battery")
ax.set_xlabel("Battery ID")
ax.set_ylabel("Discharge cycles")
plt.tight_layout()
plt.show()